In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# 1. Load Data
data = [
    [25000,120,1,'Salaried','Approved'],[32000,150,1,'Salaried','Approved'],[18000,100,0,'Self_Employed','Rejected'],[45000,200,1,'Business','Approved'],[22000,180,0,'Salaried','Rejected'],[55000,250,1,'Business','Approved'],[28000,130,1,'Self_Employed','Approved'],[15000,90,0,'Unemployed','Rejected'],[38000,175,1,'Salaried','Approved'],[20000,160,0,'Self_Employed','Rejected'],
    [60000,300,1,'Business','Approved'],[27000,140,1,'Salaried','Approved'],[19000,110,0,'Unemployed','Rejected'],[42000,210,1,'Salaried','Approved'],[24000,190,0,'Self_Employed','Rejected'],[50000,230,1,'Business','Approved'],[31000,145,1,'Salaried','Approved'],[17000,125,0,'Unemployed','Rejected'],[47000,220,1,'Self_Employed','Approved'],[23000,170,0,'Salaried','Rejected'],
    [65000,350,1,'Business','Approved'],[29000,135,1,'Salaried','Approved'],[16000,105,0,'Unemployed','Rejected'],[40000,190,1,'Salaried','Approved'],[21000,155,0,'Self_Employed','Rejected'],[52000,240,1,'Business','Approved'],[34000,160,1,'Salaried','Approved'],[14000,85,0,'Unemployed','Rejected'],[44000,205,1,'Self_Employed','Approved'],[26000,185,0,'Salaried','Rejected'],
    [58000,280,1,'Business','Approved'],[30000,150,1,'Salaried','Approved'],[17500,115,0,'Unemployed','Rejected'],[41000,195,1,'Salaried','Approved'],[22500,165,0,'Self_Employed','Rejected'],[54000,260,1,'Business','Approved'],[36000,170,1,'Salaried','Approved'],[15500,95,0,'Unemployed','Rejected'],[46000,215,1,'Self_Employed','Approved'],[23500,175,0,'Salaried','Rejected'],
    [62000,320,1,'Business','Approved'],[33000,155,1,'Salaried','Approved'],[18500,120,0,'Unemployed','Rejected'],[43000,200,1,'Salaried','Approved'],[21500,150,0,'Self_Employed','Rejected'],[51000,235,1,'Business','Approved'],[35000,165,1,'Salaried','Approved'],[16500,100,0,'Unemployed','Rejected'],[48000,225,1,'Self_Employed','Approved'],[24500,180,0,'Salaried','Rejected'],
    [57000,275,1,'Business','Approved'],[30500,148,1,'Salaried','Approved'],[19500,118,0,'Unemployed','Rejected'],[39500,188,1,'Salaried','Approved'],[20500,158,0,'Self_Employed','Rejected'],[53000,245,1,'Business','Approved'],[37000,172,1,'Salaried','Approved'],[14500,88,0,'Unemployed','Rejected'],[45500,212,1,'Self_Employed','Approved'],[25500,182,0,'Salaried','Rejected']
]
columns = ['Applicant_Income','Loan_Amount','Credit_History','Employment_Type','Loan_Status']
df = pd.DataFrame(data, columns=columns)
print('Dataset size:', len(df))

# 2. Preprocessing & One-Hot Encoding
X = df[['Applicant_Income','Loan_Amount','Credit_History','Employment_Type']]
y = df['Loan_Status']
X = pd.get_dummies(X, columns=['Employment_Type'], dtype=int)

# 3. Train/Validation/Test Splits
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full)

# 4. Tune k for k-NN using Validation set
scaler_val = StandardScaler()
X_train_scaled = scaler_val.fit_transform(X_train)
X_valid_scaled = scaler_val.transform(X_valid)

validation_scores = []
k_values = range(1, 21)
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    score = accuracy_score(y_valid, knn.predict(X_valid_scaled))
    validation_scores.append(score)

best_k = k_values[validation_scores.index(max(validation_scores))]
print('Best k selected:', best_k)

# 5. Final Model Training on full training set
decision_tree = DecisionTreeClassifier(max_depth=4, random_state=42)
decision_tree.fit(X_train_full, y_train_full)

random_forest = RandomForestClassifier(n_estimators=100, random_state=42)
random_forest.fit(X_train_full, y_train_full)

scaler = StandardScaler()
X_train_full_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

knn_final = KNeighborsClassifier(n_neighbors=best_k)
knn_final.fit(X_train_full_scaled, y_train_full)

# 6. Evaluation and Predictions on Test data
predictions = {
    'Decision Tree': decision_tree.predict(X_test),
    'k-NN': knn_final.predict(X_test_scaled),
    'Random Forest': random_forest.predict(X_test)
}

for name, predicted in predictions.items():
    print(f'\n{name} Performance:')
    print('Accuracy:', accuracy_score(y_test, predicted)*100, "%")
    print(classification_report(y_test, predicted, zero_division=0))

# 7. Visualization of Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (name, predicted) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, predicted)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Approved', 'Rejected'])
    disp.plot(ax=axes[i], cmap='Blues')
    axes[i].set_title(f'{name} Confusion Matrix')
plt.tight_layout()
plt.show()

# 8. Plot Decision Tree Structure
plt.figure(figsize=(18, 8))
plot_tree(decision_tree, feature_names=X.columns, class_names=decision_tree.classes_, filled=True, rounded=True)
plt.title('Decision Tree structure')
plt.show()

# 9. Inference on New Applicant
new_applicant = pd.DataFrame({'Applicant_Income': [42000], 'Loan_Amount': [190], 'Credit_History': [1], 'Employment_Type': ['Salaried']})
new_applicant_encoded = pd.get_dummies(new_applicant, columns=['Employment_Type'], dtype=int)
new_applicant_encoded = new_applicant_encoded.reindex(columns=X.columns, fill_value=0)

print('\n--- New Applicant Prediction ---')
print('Decision Tree Prediction:', decision_tree.predict(new_applicant_encoded)[0])
print('k-NN Prediction:', knn_final.predict(scaler.transform(new_applicant_encoded))[0])
print('Random Forest Prediction:', random_forest.predict(new_applicant_encoded)[0])